In [1]:
import sqlite3
import pandas as pd
import torch
from transformers import pipeline
from tqdm.notebook import tqdm

# 1. Attiviamo l'accelerazione M2 (Metal) se disponibile, altrimenti CPU
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Hardware in uso: {device}")

# 2. Attiviamo la barra di caricamento per pandas
tqdm.pandas(desc="Analisi Sentiment Longformer")

Hardware in uso: mps


In [2]:
db_path = '../data/document_analysis.db' # Assicurati che il percorso sia corretto
conn = sqlite3.connect(db_path)

# Estraiamo il testo completo solo per i documenti utili
query_docs = """
SELECT DISTINCT d.doc_id, d.full_text 
FROM documents d
JOIN rdf_triples r ON d.doc_id = r.doc_id
WHERE d.full_text IS NOT NULL AND d.full_text != ''
"""
df_docs = pd.read_sql_query(query_docs, conn)

print(f"Totale documenti da analizzare: {len(df_docs)}")

Totale documenti da analizzare: 18125


In [3]:
# Carichiamo il Longformer (fino a 4096 token)
sentiment_pipeline = pipeline(
    "sentiment-analysis", 
    model="spacesedan/sentiment-analysis-longformer", 
    tokenizer="spacesedan/sentiment-analysis-longformer",
    truncation=True, 
    max_length=4096, 
    device=0 if torch.backends.mps.is_available() else -1
)

def analizza_sentiment_con_neutri(text):
    if not text or pd.isna(text):
        return 0.0
    
    try:
        # Il modello restituisce una label (es. 'LABEL_1') e uno score (da 0.5 a 1.0)
        res = sentiment_pipeline(text)[0]
        label = str(res['label']).upper()
        score = float(res['score'])
        
        # LOGICA NEUTRI: Se il modello ha una confidenza bassa (es. sotto il 65%),
        # significa che il documento non è marcatamente né positivo né negativo.
        if score < 0.65:
            return 0.0  # Neutro
            
        # Se invece è sicuro, assegniamo il segno corretto
        # Solitamente '1' o 'POS' indicano positivo
        if '1' in label or 'POS' in label:
            return score
        else:
            return -score
            
    except Exception as e:
        # Se un documento ha caratteri strani e fa crashare la riga, diventa neutro
        return 0.0

Loading weights:   0%|          | 0/273 [00:00<?, ?it/s]

In [4]:
# Calcoliamo il sentiment. Vedrai la barra di avanzamento scorrere.
df_docs['doc_sentiment'] = df_docs['full_text'].progress_apply(analizza_sentiment_con_neutri)

# Contiamo quanti Positivi, Negativi e Neutri abbiamo trovato
def categorizza(val):
    if val == 0.0: return 'Neutro'
    elif val > 0: return 'Positivo'
    else: return 'Negativo'

print("\nDistribuzione del Sentiment sui Documenti:")
print(df_docs['doc_sentiment'].apply(categorizza).value_counts())

Analisi Sentiment Longformer:   0%|          | 0/18125 [00:00<?, ?it/s]

Initializing global attention on CLS token...
Input ids are automatically padded to be a multiple of `config.attention_window`: 512



Distribuzione del Sentiment sui Documenti:
doc_sentiment
Neutro      15199
Negativo     2888
Positivo       38
Name: count, dtype: int64


In [5]:
# 1. Recuperiamo tutte le triplette
query_triples = "SELECT doc_id, actor AS source, target FROM rdf_triples"
df_triples = pd.read_sql_query(query_triples, conn)
conn.close()

# 2. Uniamo le triplette al sentiment calcolato
df_merged = df_triples.merge(df_docs[['doc_id', 'doc_sentiment']], on='doc_id', how='inner')

# 3. Creiamo la lista degli archi aggregando per coppia di persone
df_edges = df_merged.groupby(['source', 'target']).agg(
    weight=('doc_id', 'count'), # Quante volte interagiscono
    avg_sentiment=('doc_sentiment', 'mean') # Sentiment medio delle loro interazioni
).reset_index()

# Pulizia: togliamo le auto-referenze (es. "Epstein" -> "Epstein")
df_edges = df_edges[df_edges['source'] != df_edges['target']]

print(f"Archi totali generati per il grafo: {len(df_edges)}")
display(df_edges.head())

Archi totali generati per il grafo: 67428


,source,target,weight,avg_sentiment
0,Edward Snowden,The Nation,1,0.0
1,/ronic Times,"""Quaaludes Ends""",1,0.0
2,10th Circuit Court,victim’s right to attend a trial,1,0.0
3,10thAmend,ooohouchburn,1,0.0
4,11 SLOANS CURVE LLC,SLOANS CURVE townhouse unit 11,1,0.0


In [6]:
# Salviamo in CSV
output_path = '../data/edges_sentiment.csv'
df_edges.to_csv(output_path, index=False)
print(f"Finito! File generato con successo in: {output_path}")

Finito! File generato con successo in: ../data/edges_sentiment.csv
